# Instructor Live-Coding Demos (`4S4TWS_21`)
## 4 Bite-Sized Classroom Walkthroughs (Sessions `014`–`017`)

**Instructor:** Chouaieb Nemri  
Run each cell live on screen during the 15-minute live-coding window before sending students into their pod breakout rooms:
- **Demo 1 (Session `014`, `09:15`):** Why Free-Text Prompts Fail vs. Guaranteed Pydantic JSON
- **Demo 2 (Session `014`, `09:50`):** Watching the 4-Step Tool-Calling Handshake in Slow Motion
- **Demo 3 (Session `015`, `15:00`):** Why Basic RAG Fails on Greenwashing Claims vs. Multi-Step Agentic Search
- **Demo 4 (Session `017`, `14:55`):** Blocking a Prompt Injection & Unauthorized €5M Transfer with 10 Lines of Python


In [1]:
# DEMO 1 (Session 014): Free-Text Guessing vs. Guaranteed Pydantic Output
from typing import Literal
from pydantic import BaseModel, Field

class CarbonTaxQuote(BaseModel):
    supplier_id: str
    total_scope1_2_tco2e: float
    cbam_tax_eur: float
    risk_tier: Literal["LOW_RISK", "MEDIUM_RISK", "HIGH_RISK"] = Field(
        description="HIGH_RISK if emissions > 25,000 tCO2e without SBTi"
    )

# Exact calculation in Python + validated Pydantic contract:
scope1, scope2, price = 68400, 22100, 85.0
quote = CarbonTaxQuote(
    supplier_id="SUP-002",
    total_scope1_2_tco2e=scope1 + scope2,
    cbam_tax_eur=(scope1 + scope2) * price,
    risk_tier="HIGH_RISK",
)
print("Validated Pydantic JSON ready for downstream APIs:")
print(quote.model_dump_json(indent=2))


Validated Pydantic JSON ready for downstream APIs:
{
  "supplier_id": "SUP-002",
  "total_scope1_2_tco2e": 90500.0,
  "cbam_tax_eur": 7692500.0,
  "risk_tier": "HIGH_RISK"
}


In [2]:
# DEMO 2 (Session 014): The 4-Step Tool-Calling Handshake & Self-Healing Error Return
MINI_DB = {
    "SUP-001": {"name": "RhôneAlpin GreenSteel SA", "scope1_2_tco2e": 17300},
    "SUP-002": {"name": "BalkanFerro Heavy Works", "scope1_2_tco2e": 90500},
}

def get_supplier(supplier_id: str) -> dict:
    """Return supplier data, or a structured list of valid IDs if not found."""
    if supplier_id not in MINI_DB:
        return {
            "status": "NOT_FOUND",
            "requested_id": supplier_id,
            "valid_ids": list(MINI_DB.keys()),
            "hint": "Check the supplier ID and retry with one of valid_ids.",
        }
    return {"status": "OK", "supplier_id": supplier_id, **MINI_DB[supplier_id]}

print("1. When the model asks for a non-existent supplier (SUP-999):")
print("   ->", get_supplier("SUP-999"))
print("2. When the model self-corrects and queries SUP-001:")
print("   ->", get_supplier("SUP-001"))


1. When the model asks for a non-existent supplier (SUP-999):
   -> {'status': 'NOT_FOUND', 'requested_id': 'SUP-999', 'valid_ids': ['SUP-001', 'SUP-002'], 'hint': 'Check the supplier ID and retry with one of valid_ids.'}
2. When the model self-corrects and queries SUP-001:
   -> {'status': 'OK', 'supplier_id': 'SUP-001', 'name': 'RhôneAlpin GreenSteel SA', 'scope1_2_tco2e': 17300}


In [3]:
# DEMO 3 (Session 015): Basic 1-Shot Search vs. Multi-Step Agentic Verification
def verify_citations(agent_cited_ids: list[str], retrieved_tool_ids: set[str]) -> dict:
    hallucinated = [cid for cid in agent_cited_ids if cid not in retrieved_tool_ids]
    return {
        "all_citations_verified": len(hallucinated) == 0,
        "verified_ids": [cid for cid in agent_cited_ids if cid in retrieved_tool_ids],
        "hallucinated_ids": hallucinated,
    }

retrieved = {"REG-EU-GCD-01", "CLAIM-AERO-101"}
print("Checking faithful agent response :", verify_citations(["REG-EU-GCD-01", "CLAIM-AERO-101"], retrieved))
print("Catching invented legal article  :", verify_citations(["REG-EU-GCD-01", "EU-FAKE-ART-99"], retrieved))


Checking faithful agent response : {'all_citations_verified': True, 'verified_ids': ['REG-EU-GCD-01', 'CLAIM-AERO-101'], 'hallucinated_ids': []}
Catching invented legal article  : {'all_citations_verified': False, 'verified_ids': ['REG-EU-GCD-01'], 'hallucinated_ids': ['EU-FAKE-ART-99']}


In [4]:
# DEMO 4 (Session 017): Deterministic Python before_tool_callback Gate
def before_tool_callback(tool_name: str, tool_args: dict, user_role: str) -> dict | None:
    """Return None to allow tool execution, or a dict to block it before it runs."""
    if tool_name == "approve_wire_transfer":
        if user_role != "COMMITTEE_CHAIR":
            return {"status": "BLOCKED", "reason": f"Role '{user_role}' cannot approve wire transfers."}
        if float(tool_args.get("amount_eur", 0)) > 2_000_000:
            return {"status": "BLOCKED", "reason": "Transfers above €2,000,000 require dual human board sign-off."}
    return None

print("Attempt 1 (Analyst requests €500k)  :", before_tool_callback("approve_wire_transfer", {"amount_eur": 500_000}, "ANALYST"))
print("Attempt 2 (Chair requests €5.5M)    :", before_tool_callback("approve_wire_transfer", {"amount_eur": 5_500_000}, "COMMITTEE_CHAIR"))
print("Attempt 3 (Chair requests €1.2M)    :", before_tool_callback("approve_wire_transfer", {"amount_eur": 1_200_000}, "COMMITTEE_CHAIR") or "ALLOWED -> Executing tool")


Attempt 1 (Analyst requests €500k)  : {'status': 'BLOCKED', 'reason': "Role 'ANALYST' cannot approve wire transfers."}
Attempt 2 (Chair requests €5.5M)    : {'status': 'BLOCKED', 'reason': 'Transfers above €2,000,000 require dual human board sign-off.'}
Attempt 3 (Chair requests €1.2M)    : ALLOWED -> Executing tool
